# TorGen MDN-CVAE: Mixture Density Point Process

Conditional VAE with mixture density output for tornado outbreak generation.

In [ ]:
%matplotlib inline
!pip install -q --no-cache-dir --force-reinstall git+https://github.com/jcaramichaellehigh/TorGen.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training.mdn_config import MDNTrainConfig
from torgen.training.mdn_trainer import train_mdn

config = MDNTrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints_mdn",
)

trainer = train_mdn(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
h = trainer.loss_history
print(f"Final total: {h['train_total'][-1]:.4f}")
print(f"Final spatial: {h['train_spatial'][-1]:.4f}")
print(f"Final count (\u039b): {h['train_count'][-1]:.2f}")
print(f"Final EF: {h['train_ef'][-1]:.4f}")
print(f"Final KL: {h['train_kl'][-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

h = trainer.loss_history
epochs = range(1, len(h["train_total"]) + 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(epochs, h["train_total"], label="Train")
axes[0].plot(epochs, h["val_total"], label="Val")
axes[0].set_title("Total Loss")
axes[0].legend()

axes[1].plot(epochs, h["train_spatial"], label="Train")
axes[1].plot(epochs, h["val_spatial"], label="Val")
axes[1].set_title("Spatial NLL")
axes[1].legend()

axes[2].plot(epochs, h["train_count"], label="Train")
axes[2].plot(epochs, h["val_count"], label="Val")
axes[2].set_title("Count (\u039b)")
axes[2].legend()

axes[3].plot(epochs, h["train_kl"], label="Train KL")
axes[3].plot(epochs, h["val_kl"], label="Val KL")
axes[3].set_title("KL Divergence")
axes[3].legend()

for ax in axes:
    ax.set_xlabel("Epoch")
fig.tight_layout()
plt.show()

## Diversity: Multiple Z Draws per Day

Gray ellipses show 1\u03c3 contours of active mixture components. Colored dots are sampled tornadoes.

In [ ]:
import os
import torch
import numpy as np
from matplotlib.patches import Ellipse
from torgen.data.dataset import _load_pt, coords_to_bearing_length
from torgen.sampling import sample_outbreak
from torgen.viz.plots import _ef_color


def plot_mdn_sample(ax, mdn_params, sample, title=""):
    """Plot sampled tornadoes + component ellipses."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("easting")
    ax.set_ylabel("northing")

    pi = mdn_params["pi"][0].cpu().numpy()
    mu = mdn_params["mu"][0].cpu().numpy()
    L = mdn_params["L"][0].cpu().numpy()

    # Draw 1-sigma ellipses for active components
    for k in range(len(pi)):
        if pi[k] < 0.1:
            continue
        cov = L[k] @ L[k].T
        vals, vecs = np.linalg.eigh(cov)
        angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
        w, h = 2 * np.sqrt(np.maximum(vals, 1e-8))
        ell = Ellipse(xy=mu[k], width=w, height=h, angle=angle,
                      alpha=0.15, color="gray", linewidth=1)
        ax.add_patch(ell)
        # Show weight as text
        ax.text(mu[k, 0], mu[k, 1], f"{pi[k]:.1f}", fontsize=6,
                ha="center", va="center", color="gray", alpha=0.7)

    # Plot sampled tornadoes
    if sample["count"] > 0:
        coords = sample["coords"].cpu().numpy()
        efs = sample["ef"].cpu().numpy()
        for i in range(len(coords)):
            color = _ef_color(int(efs[i]))
            ax.plot(coords[i, 0], coords[i, 1], "o", color=color,
                    markersize=4, alpha=0.8)


def plot_gt_points(ax, tracks, title=""):
    """Plot GT tornado genesis points."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("easting")
    ax.set_ylabel("northing")
    if tracks.shape[0] == 0:
        ax.text(0.5, 0.5, "No tornadoes", ha="center", va="center",
                fontsize=10, color="gray", transform=ax.transAxes)
        return
    coords = tracks[:, :2].numpy()
    efs = tracks[:, 5].numpy()
    for i in range(len(coords)):
        color = _ef_color(int(efs[i]))
        ax.plot(coords[i, 0], coords[i, 1], "o", color=color,
                markersize=4, alpha=0.8)


days = ["2011-04-27.pt", "2019-05-20.pt", "2023-03-31.pt"]
n_samples = 8
n_rows = len(days)
n_cols = 1 + n_samples
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes[None, :]

trainer.model.eval()

for row, day_file in enumerate(days):
    path = os.path.join(config.local_cache_dir, day_file)
    sample_data = _load_pt(path)
    gt_tracks = coords_to_bearing_length(sample_data["tracks"])
    date = sample_data.get("date", day_file.replace(".pt", ""))
    n_gt = gt_tracks.shape[0]

    plot_gt_points(axes[row, 0], gt_tracks,
                   title=f"GT \u2014 {date}\n({n_gt} tracks)")

    wx = sample_data["wx"].unsqueeze(0).to(trainer.device)

    for col in range(n_samples):
        with torch.no_grad():
            out = trainer.model.generate(wx)
        mdn_params = out["mdn_params"]
        s = sample_outbreak(mdn_params, batch_idx=0)
        plot_mdn_sample(axes[row, 1 + col], mdn_params, s,
                        title=f"Sample {col+1}\n({s['count']} tracks)")

fig.tight_layout()
plt.show()

## Count and EF Distribution Comparison

In [ ]:
from torgen.data.dataset import TornadoDataset

ds = TornadoDataset(config.local_cache_dir, split="test")
gt_counts = []
gen_counts = []
gt_ef = []
gen_ef = []

trainer.model.eval()
for i in range(len(ds)):
    sample_data = ds[i]
    tracks = sample_data["tracks"]
    gt_counts.append(tracks.shape[0])
    if tracks.shape[0] > 0:
        gt_ef.extend(tracks[:, 5].long().tolist())

    wx = sample_data["wx"].unsqueeze(0).to(trainer.device)
    for _ in range(config.n_eval_samples):
        with torch.no_grad():
            out = trainer.model.generate(wx)
        s = sample_outbreak(out["mdn_params"])
        gen_counts.append(s["count"])
        if s["count"] > 0:
            gen_ef.extend(s["ef"].tolist())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count distribution
max_count = max(max(gt_counts, default=0), max(gen_counts, default=0), 1)
bins = np.arange(-0.5, min(max_count + 1.5, 50), 1)
axes[0].hist(gt_counts, bins=bins, alpha=0.6, label="Ground Truth", density=True)
axes[0].hist(gen_counts, bins=bins, alpha=0.6, label="Generated", density=True)
axes[0].set_xlabel("Track Count")
axes[0].set_ylabel("Density")
axes[0].set_title("Track Count Distribution")
axes[0].legend()

# EF distribution
ef_labels = [f"EF{i}" for i in range(6)]
x = np.arange(6)
w = 0.35
gt_hist = np.bincount(gt_ef, minlength=6)[:6].astype(float)
gen_hist = np.bincount(gen_ef, minlength=6)[:6].astype(float)
gt_hist /= gt_hist.sum() if gt_hist.sum() > 0 else 1
gen_hist /= gen_hist.sum() if gen_hist.sum() > 0 else 1

axes[1].bar(x - w/2, gt_hist, w, label="GT", alpha=0.8)
axes[1].bar(x + w/2, gen_hist, w, label="Generated", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(ef_labels)
axes[1].set_title("EF Distribution")
axes[1].legend()

fig.tight_layout()
plt.show()